# SFT benchmark: one epoch with Meep², Unsloth and TRL

<small>Copyright Alejandro Martínez Corriá and the Thinkube contributors. SPDX-License-Identifier: Apache-2.0</small>

This notebook runs the comparison the README publishes: one epoch of LoRA on no_robots, three engines, the same batches in the same order, and the held-out loss of every trained adapter measured by the same evaluation.

| Arm | What it is |
|---|---|
| `trl` | Transformers, PEFT and TRL at their default settings: padded batches, every layer checkpointed |
| `unsloth` | Unsloth at its fastest: gradient checkpointing off |
| `meepsquared` | Meep²'s auto mode: it measures the free memory and checkpoints only the layers it must |

Every arm takes its training decisions (batch, LoRA rank and targets, optimizer, learning rate, schedule) from Meep²'s plan, which the notebook makes first. At the end, each adapter is evaluated through TRL's evaluation on 400 conversations it never trained on: on Gemma 4 and Qwen 3.5, each engine's own evaluation reads differently, Unsloth's lower even before training, so the engines' own numbers are not compared.

One epoch takes about 25 to 140 minutes per arm and model on a DGX Spark; set `STEPS = 30` for a quick check of the setup.

## Settings

`MODEL_SOURCE` says where the models come from: `"huggingface"` downloads them from the Hub by their id; `"thinkube"`, on [Thinkube](https://thinkube.org), reads them from the cluster's model mirror.

`UNSLOTH_PYTHON` is the Python of an environment with Unsloth installed: Unsloth patches Transformers and TRL when it is imported, and needs older versions of both (Unsloth 2026.10.7 ran with transformers 5.17.0 and TRL 0.29.1), so its arm runs in an environment of its own. Leave it `None` to run the other two arms only.

A notebook job can override any of these with its parameters.

In [ ]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")   # "huggingface" or "thinkube"
MODELS = PARAMETERS.get("MODELS", ["google/gemma-4-E4B-it", "Qwen/Qwen3.5-9B"])
ARMS = PARAMETERS.get("ARMS", ["meepsquared", "unsloth", "trl"])
STEPS = PARAMETERS.get("STEPS", 1137)          # one epoch of no_robots: 9,100 conversations, 8 per step
EVAL_ROWS = PARAMETERS.get("EVAL_ROWS", 400)   # the last conversations of the data, never trained on
EVAL_EVERY = PARAMETERS.get("EVAL_EVERY", 100)
SEED = PARAMETERS.get("SEED", 42)
UNSLOTH_PYTHON = PARAMETERS.get("UNSLOTH_PYTHON", None)
BENCHMARKS = PARAMETERS.get("BENCHMARKS", None)  # the benchmarks folder of meepsquared; None for the current folder

## The environment

The `meepsquared` and `trl` arms run with this kernel's Python, the `unsloth` arm with `UNSLOTH_PYTHON`. Results go to `results/sft-epoch.jsonl`, plans to `results/plans/`, adapters to `results/adapters/`.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

BENCHMARKS = Path(BENCHMARKS) if BENCHMARKS else Path.cwd()
if not (BENCHMARKS / "sft.py").is_file():
    raise FileNotFoundError(f"sft.py is not in {BENCHMARKS}: run this notebook from the benchmarks folder of meepsquared, "
                            "or set BENCHMARKS to that folder")
if "unsloth" in ARMS and UNSLOTH_PYTHON is None:
    raise ValueError("ARMS has unsloth and UNSLOTH_PYTHON is None: give the Python of an environment with Unsloth, "
                     "or leave unsloth out of ARMS")
RESULTS = BENCHMARKS / "results"
RESULTS_FILE = RESULTS / "sft-epoch.jsonl"
PLANS, ADAPTERS = RESULTS / "plans", RESULTS / "adapters"
PLANS.mkdir(parents=True, exist_ok=True)
ADAPTERS.mkdir(parents=True, exist_ok=True)
NAME = {m: m.split("/")[-1] for m in MODELS}


def run(python, args):
    """Run a benchmark script, printing its output as it comes; a failure stops the notebook with the script's error."""
    p = subprocess.Popen([python, "-u", *args], cwd=BENCHMARKS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{args[0]} exited with {p.returncode}")

## The plans

Meep² plans each model's training on this machine: the batch, the checkpointing, the LoRA settings and the learning rate. Every arm takes the training decisions from this plan, and trains the same batches in the same order.

In [ ]:
for model in MODELS:
    run(sys.executable, ["plan.py", "--model", model, "--method", "lora", "--source", MODEL_SOURCE, "--steps", str(STEPS),
                         "--rows", str(8 * STEPS), "--out", str(PLANS / f"{NAME[model]}-epoch.json")])

## Training

Each arm trains each model for `STEPS` steps, measures the held-out loss every `EVAL_EVERY` steps with its own evaluation, and saves its adapter. The first ten steps are not counted in the speed: the kernels compile, and on a machine whose memory the GPU shares, the system is still caching the model's files.

In [ ]:
def train(model, arm):
    common = ["--model", model, "--method", "lora", "--source", MODEL_SOURCE, "--steps", str(STEPS), "--warmup", "10",
              "--seed", str(SEED), "--plan", str(PLANS / f"{NAME[model]}-epoch.json"), "--eval-rows", str(EVAL_ROWS),
              "--eval-every", str(EVAL_EVERY), "--save-adapter", str(ADAPTERS / f"{NAME[model]}-{arm}"),
              "--results", str(RESULTS_FILE), "--label", f"{NAME[model]}-{arm}"]
    if arm == "meepsquared":
        run(sys.executable, ["sft.py", "--arm", "meepsquared", "--attn", "auto", "--checkpointing", "plan", *common])
    elif arm == "unsloth":
        # Unsloth's compiler recompiled Qwen 3.5's linear-attention layers at every step in this environment
        env = ["run_env.py", "UNSLOTH_COMPILE_DISABLE=1", "--"] if "Qwen3.5" in model else []
        run(UNSLOTH_PYTHON, [*env, "sft.py", "--arm", "unsloth", "--checkpointing", "false", *common])
    else:
        run(sys.executable, ["sft.py", "--arm", "trl", *common])


for model in MODELS:
    for arm in ARMS:
        train(model, arm)

## The same evaluation for every adapter

Each saved adapter is loaded into the `trl` arm's model and evaluated on the same held-out conversations. Unsloth saves Qwen 3.5's adapter under the names of its multimodal wrapper (`model.language_model.layers…`); the text model calls the same layers `model.layers…`, so those names are rewritten in a copy, with the weights unchanged. The loading stops with an error if any weight has no module, so an adapter that did not load can never be reported.

In [ ]:
import struct


def text_model_names(adapter: Path) -> Path:
    """The adapter itself, or a copy whose weight names drop the multimodal wrapper's language_model."""
    weights = adapter / "adapter_model.safetensors"
    with open(weights, "rb") as f:
        size = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(size))
        data = f.read()
    if not any(".language_model." in k for k in header):
        return adapter
    renamed = {k.replace("model.language_model.", "model.", 1): v for k, v in header.items()}
    copy = adapter.with_name(adapter.name + "-text-names")
    copy.mkdir(exist_ok=True)
    (copy / "adapter_config.json").write_text((adapter / "adapter_config.json").read_text())
    encoded = json.dumps(renamed, separators=(",", ":")).encode()
    encoded += b" " * ((8 - len(encoded) % 8) % 8)
    with open(copy / "adapter_model.safetensors", "wb") as f:
        f.write(struct.pack("<Q", len(encoded)) + encoded + data)
    return copy


shared = {}
for model in MODELS:
    for arm in ARMS:
        adapter = text_model_names(ADAPTERS / f"{NAME[model]}-{arm}")
        out = subprocess.run([sys.executable, "-u", "sft.py", "--arm", "trl", "--model", model, "--method", "lora",
                              "--source", MODEL_SOURCE, "--steps", str(STEPS), "--seed", str(SEED),
                              "--plan", str(PLANS / f"{NAME[model]}-epoch.json"), "--eval-rows", str(EVAL_ROWS),
                              "--eval-only", "--adapter", str(adapter)],
                             cwd=BENCHMARKS, capture_output=True, text=True)
        line = next((l for l in out.stdout.splitlines() if l.startswith("RESULT eval-only")), None)
        if out.returncode != 0 or line is None:
            raise RuntimeError(f"the evaluation of {adapter} failed:\n{out.stdout[-3000:]}{out.stderr[-3000:]}")
        shared[(model, arm)] = json.loads(line.split(" ", 2)[2])["eval_loss"]
        print(f"{NAME[model]:24} {arm:12} held-out loss {shared[(model, arm)]:.4f}")

## Results

Minutes of training (evaluation not counted), the peak GPU memory, and the held-out loss of each adapter from the same evaluation.

In [ ]:
import pandas as pd

rows = [json.loads(line) for line in RESULTS_FILE.read_text().splitlines()]
table = []
for model in MODELS:
    for arm in ARMS:
        r = [x for x in rows if x["arm"] == f"{NAME[model]}-{arm}" and x["steps"] == STEPS][-1]
        table.append({"model": NAME[model], "engine": arm, "minutes": round(r["quality"][-1]["training_seconds"] / 60, 1),
                      "peak GiB": max(r["peak_gib"]), "held-out loss": round(shared[(model, arm)], 4)})
display(pd.DataFrame(table).set_index(["model", "engine"]))